# Milestone 3 — Exploratory Data Analysis (EDA)
**Project:** IBM Telco Customer Churn Prediction  
**Author:** AI Pair Programmer / Antigravity  
**Date:** July 2026  

---

## Executive Summary

This notebook presents a production-oriented Exploratory Data Analysis (EDA) of the IBM Telco Customer Churn dataset. Unlike an unstructured exploratory notebook, this analysis is structured around concrete engineering and modelling objectives: validating schema contract assumptions, assessing data quality, checking for data leakage, reviewing fairness-relevant demographic features, and establishing downstream preprocessing and model architecture decisions.

### Key EDA Findings
1. **Dataset Dimensions & Integrity:** 7,043 customer snapshot records across 21 features with 0 duplicate rows and 0 duplicate `customerID`s.
2. **Missing Value Nuance:** Exactly 11 records have blank `TotalCharges` values. Analysis confirms that 100% of these records correspond to brand-new customers with `tenure == 0`. They can be safely imputed with `0.0`.
3. **Target Imbalance:** Target variable `Churn` contains 1,869 churned customers (26.54%) and 5,174 retained customers (73.46%), yielding an imbalance ratio of **2.77 : 1**. Precision-Recall AUC (PR-AUC) and recall at top capacity are appropriate evaluation metrics instead of accuracy.
4. **Primary Churn Drivers:**
   - **Contract Type:** Month-to-month contract customers churn at **42.71%**, compared to 11.27% for 1-year and 2.83% for 2-year.
   - **Internet Tier:** Fiber optic customers churn at **41.89%**, compared to 18.96% for DSL and 7.40% for non-internet.
   - **Payment Channel:** Electronic check users churn at **45.29%**, vs ~15.5% for automatic payment channels.
   - **Customer Tenure:** Early tenure customers (0-12 months) churn at **47.44%**, dropping below 7% for long-tenure (>60 months).
5. **Collinearity & Discrepancy:** `TotalCharges` is highly correlated with `tenure` ($r = 0.83$) and `MonthlyCharges` ($r = 0.65$). The mean absolute discrepancy between `TotalCharges` and $\text{tenure} \times \text{MonthlyCharges}$ is **$45.09**, reflecting mid-contract price changes and service modifications.
6. **Fairness & Leakage Review:**
   - `gender` displays near-zero churn rate disparity (Female 26.92% vs Male 26.16%) and must be excluded from feature inputs.
   - `SeniorCitizen` customers exhibit higher churn (41.68% vs 23.61%), but are excluded from model inputs per `SYSTEM_DESIGN.md` guidelines to prevent age bias, remaining reserved for governed sub-population evaluation.
   - `customerID` is 100% unique and carries no predictive information. No post-churn outcome features exist.


## 1. Environment Setup & Reusable Data Load

Reusable data validation and EDA utilities are imported from `src/churn_prediction/data/` and `src/churn_prediction/eda/` to ensure no production logic resides in the notebook.


In [ ]:
from pathlib import Path

import pandas as pd

from churn_prediction.data.validator import parse_total_charges
from churn_prediction.eda import (
    assess_leakage_and_fairness,
    get_categorical_summary,
    get_charge_discrepancy_analysis,
    get_dataset_overview,
    get_numeric_correlations,
    get_numeric_summary,
    get_target_distribution,
)

data_path = (
    Path("../Telco-Customer-Churn.csv")
    if Path("../Telco-Customer-Churn.csv").exists()
    else Path("Telco-Customer-Churn.csv")
)
raw_df = pd.read_csv(data_path)
df, parsing_errors = parse_total_charges(raw_df)
print(f"Loaded {len(df)} rows and {len(df.columns)} columns.")

## 2. Dataset Dimensions, Schema & Quality Audit

We verify dataset structure, data types, missing value distribution, and duplicate records.


In [ ]:
overview = get_dataset_overview(df)
print("Total Rows:", overview["total_rows"])
print("Total Columns:", overview["total_cols"])
print("Duplicate Rows:", overview["duplicate_rows"])
print("Duplicate Customer IDs:", overview["duplicate_customer_ids"])
print("Missing Value Counts:", overview["missing_counts"])

### Analysis of Missing `TotalCharges` Values
All 11 missing `TotalCharges` values occur in rows where `tenure == 0`. These represent newly onboarded customers who have not completed a full billing cycle.


In [ ]:
cols = [
    "customerID",
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "Churn",
]
blank_tc_df = df[df["TotalCharges"].isna()]
print("Tenure values for missing TotalCharges rows:")
print(blank_tc_df[cols])

## 3. Target Distribution & Class Imbalance Analysis

Evaluating the binary target variable `Churn`.


In [ ]:
target_dist = get_target_distribution(df)
print("Class Counts:", target_dist["counts"])
print("Class Percentages:", target_dist["percentages"])
print(f"Overall Churn Rate: {target_dist['churn_rate'] * 100:.2f}%")
imb = target_dist["imbalance_ratio"]
print(f"Imbalance Ratio (No/Yes): {imb:.2f}")

![Target Distribution](../docs/images/target_distribution.png)


## 4. Numeric Feature Distributions & Outlier Assessment

Analyzing summary statistics, spread, skewness, and IQR outlier boundaries for `tenure`, `MonthlyCharges`, and `TotalCharges`.


In [ ]:
numeric_summary = get_numeric_summary(df)
print(numeric_summary.to_string())

![Numeric Distributions](../docs/images/numeric_distributions.png)


## 5. Charge Discrepancy & Pearson Correlation Analysis

Evaluating the mathematical relationship between `TotalCharges` and `tenure * MonthlyCharges`, along with feature collinearity.


In [ ]:
disc_analysis = get_charge_discrepancy_analysis(df)
for metric, val in disc_analysis.items():
    print(f"{metric}: {val}")

corr_matrix = get_numeric_correlations(df)
print("\nPearson Correlation Matrix:")
print(corr_matrix)

![Correlation Heatmap](../docs/images/correlation_heatmap.png)
![Tenure vs Total Charges](../docs/images/tenure_vs_charges.png)


## 6. Categorical Feature Distributions & Churn Risk Drivers

Examining category frequencies and churn rates across key contract, service, and payment dimensions.


In [ ]:
cat_cols = [
    "Contract",
    "InternetService",
    "PaymentMethod",
    "TechSupport",
]
cat_summaries = get_categorical_summary(df, categorical_cols=cat_cols)
for col_name, summary_df in cat_summaries.items():
    print(f"\n--- Feature: {col_name} ---")
    print(summary_df.to_string(index=False))

![Categorical Churn Rates](../docs/images/categorical_churn_rates.png)


## 7. Data Leakage & Sensitive Demographic Attribute Assessment

Verifying data hygiene and demographic neutrality.


In [ ]:
leakage_fairness = assess_leakage_and_fairness(df)
print("Leakage Review:")
for item in leakage_fairness["leakage_findings"]:
    print(" -", item)

print("\nFairness Review:")
for item in leakage_fairness["fairness_findings"]:
    print(" -", item)

![Fairness Attributes](../docs/images/fairness_attributes.png)


## 8. Confirmation of `SYSTEM_DESIGN.md` Assumptions & Implications

### Confirmation of System Design Assumptions
| Assumption | Status | Empirical Finding |
|---|---|---|
| 11 blank `TotalCharges` values | **Confirmed** | All 11 records have `tenure == 0`. Impute with `0.0`. |
| Target Imbalance ~26.5% | **Confirmed** | Exact churn rate is 26.54% (1,869 Yes vs 5,174 No). |
| `customerID` Non-Predictive | **Confirmed** | 100% unique hash; must be excluded from feature inputs. |
| `gender` Neutrality | **Confirmed** | Disparity is < 0.8% (Female: 26.92%, Male: 26.16%). |
| `SeniorCitizen` Higher Churn | **Confirmed** | Seniors churn at 41.68% vs 23.61%. Excluded for fairness. |
| Primary Drivers | **Confirmed** | `Contract`, `InternetService`, `tenure` are drivers. |

### Downstream Pipeline Implications (Milestone 4+)
1. **Imputation:** Impute `TotalCharges` missing values with `0.0` inside transformer (matching `tenure == 0`).
2. **Stratified Splitting:** Use stratified $k$-fold / train-test splits on `Churn` to preserve target proportion.
3. **Feature Selection:** Exclude `customerID`, `gender`, and `SeniorCitizen` from `X` feature matrix.
4. **Encoding:** One-hot encode nominal categories with `handle_unknown='ignore'`.
5. **Scaling:** Standardize/robust-scale `tenure`, `MonthlyCharges`, `TotalCharges` for baseline.
